[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/evaluations/builtin_evaluators.ipynb)

# Evaluations: Built-in & Custom Evaluators

**Score your agent's traces automatically with deterministic checks and LLM-powered judges.**

This notebook covers:
- 5 built-in deterministic evaluators (no LLM needed)
- 5 LLM-powered evaluators (requires an API key)
- Writing custom evaluators with the `@eval` decorator
- Online evaluation (auto-score production traffic)
- Batch evaluation (score existing traces)

## Setup

In [ ]:
!pip install -q 'decimalai[evals]'

In [ ]:
import os
os.environ["DECIMAL_API_KEY"] = "dai_sk_..."  # ← Your DecimalAI key
# os.environ["OPENAI_API_KEY"] = "sk-..."    # ← Needed for LLM evaluators

import decimalai
decimalai.init()

## Part 1: Deterministic Evaluators (No LLM Needed)

These run instantly, cost nothing, and are great for structural checks.

In [ ]:
from decimalai.evals import (
    eval, TraceData, EvalResult,
    json_valid, contains, not_contains, regex_match, length_check,
)

# Create a sample trace for testing
sample = TraceData(
    id="test-001",
    input="What is the return policy?",
    output='Our return policy allows returns within 30 days. Contact support@example.com for help.',
    status="success",
    agent_name="support-agent",
)

print("Sample trace:")
print(f"  Input:  {sample.input}")
print(f"  Output: {sample.output}")

In [ ]:
# ── 1. json_valid: checks if output is valid JSON ──
json_check = json_valid
result = json_check(sample)
print(f"json_valid: passed={result.passed if result else 'skipped'}")  # False — output is text

# ── 2. contains: checks if output contains a substring ──
has_policy = contains(["return"])
result = has_policy(sample)
print(f"contains('return'): passed={result.passed if result else 'skipped'}")  # True

# ── 3. not_contains: checks output doesn't contain forbidden text ──
no_pii = not_contains(["555-"])
result = no_pii(sample)
print(f"not_contains('555-'): passed={result.passed if result else 'skipped'}")  # True

# ── 4. regex_match: checks output against a regex pattern ──
has_email = regex_match(r'[\w.]+@[\w.]+')
result = has_email(sample)
print(f"regex_match(email): passed={result.passed if result else 'skipped'}")  # True

# ── 5. length_check: checks output length is within bounds ──
good_length = length_check(min_chars=20, max_chars=500)
result = good_length(sample)
print(f"length_check(20-500): passed={result.passed if result else 'skipped'}")  # True

## Part 2: Custom Evaluators with `@eval`

The `@eval` decorator turns any function into an evaluator.
Your function receives a `TraceData` object and returns:
- `bool` — pass/fail
- `float` — score between 0.0 and 1.0
- `EvalResult` — full control over score, pass/fail, and reason

In [ ]:
# ── Custom eval: simple bool ──
@eval(name="answered_question")
def check_answered(trace: TraceData) -> bool:
    """Check that the agent actually answered (not just repeated the question)."""
    return len(trace.output) > 20 and trace.input.lower() not in trace.output.lower()

result = check_answered(sample)
print(f"answered_question: passed={result.passed}, score={result.score}")


# ── Custom eval: float score ──
@eval(name="response_quality")
def quality_score(trace: TraceData) -> float:
    """Score response quality on multiple dimensions."""
    score = 0.0
    if len(trace.output) > 50: score += 0.3       # Substantial response
    if "?" not in trace.output[-20:]: score += 0.3  # Doesn't end with question
    if trace.status == "success": score += 0.4      # No errors
    return score

result = quality_score(sample)
print(f"response_quality: score={result.score}, passed={result.passed}")


# ── Custom eval: EvalResult with reason ──
@eval(name="no_hallucination")
def check_hallucination(trace: TraceData) -> EvalResult:
    """Check for common hallucination patterns."""
    hallucination_phrases = ["I think", "probably", "I'm not sure", "maybe"]
    found = [p for p in hallucination_phrases if p.lower() in trace.output.lower()]
    if found:
        return EvalResult(score=0.3, passed=False, reason=f"Hedging detected: {found}")
    return EvalResult(score=1.0, passed=True, reason="No hedging language found")

result = check_hallucination(sample)
print(f"no_hallucination: score={result.score}, passed={result.passed}, reason={result.reason}")

## Part 3: LLM-Powered Evaluators

For nuanced quality checks, DecimalAI includes LLM-as-a-judge evaluators.
These use `litellm` under the hood and support any LLM provider.

> **Requires**: `OPENAI_API_KEY` (or another LLM provider key) set in environment.

In [ ]:
# Uncomment and run if you have an OpenAI API key:

# from decimalai.evals import Relevance, Factuality, Faithfulness, Toxicity, Conciseness
#
# relevance = Relevance()       # Is the output relevant to the input?
# factuality = Factuality()     # Are the facts accurate?
# faithfulness = Faithfulness() # Does output align with retrieved context?
# toxicity = Toxicity()         # Is the output safe and appropriate?
# conciseness = Conciseness()   # Is the output concise?
#
# result = relevance(sample)
# print(f"Relevance: score={result.score}, passed={result.passed}")
#
# result = toxicity(sample)
# print(f"Toxicity: score={result.score}, passed={result.passed}")

## Part 4: Online Evaluation (Auto-Score Production Traffic)

Pass your evaluators to the `install()` function. Every trace is
automatically scored before being sent to DecimalAI.

In [ ]:
# Online eval with LangChain
# from decimalai.langchain import install
# install(
#     agent_name="support-agent",
#     evals=[check_answered, quality_score, check_hallucination],
# )
#
# # Now every trace is auto-scored!
# agent.invoke({"input": "What is your return policy?"})

print("ℹ️  Uncomment the code above in your production agent.")
print("   Every trace will be scored by your evaluators automatically.")

## Part 5: Batch Evaluation (Score Existing Traces)

Use `batch_eval` to run evaluators against traces already in DecimalAI.

In [ ]:
# Batch eval against existing traces
# from decimalai import batch_eval
#
# results = batch_eval(
#     trace_ids=["trace-001", "trace-002", "trace-003"],
#     evals=[check_answered, quality_score, check_hallucination],
# )
# print(f"Evaluated: {results['traces_evaluated']} traces")
# print(f"Summary: {results['summary']}")

print("ℹ️  Replace trace_ids with real IDs from your dashboard.")

## Summary

| Type | Examples | Cost | Speed |
|------|---------|------|-------|
| **Deterministic** | json_valid, contains, regex_match, length_check | Free | Instant |
| **Custom** | @eval decorator — bool, float, or EvalResult | Free | Instant |
| **LLM Judge** | Relevance, Factuality, Toxicity, Conciseness | LLM cost | ~1-3s |

## Next Steps

- 📖 [Quickstart](../quickstart/quickstart.ipynb) — Get your first traces
- 📖 [Manifest Changes](../version-aware-loop/manifest_change.ipynb) — Version-aware loop
- 📖 [Build Datasets](../datasets-and-training/build_sft_dataset.ipynb) — Create training data from scored traces